# Phần 1 — Câu hỏi Trắc nghiệm (MCQ)
Tính trực tiếp từ data để tìm đáp án chính xác cho 10 câu hỏi.
Mỗi câu đúng: **2 điểm** | Không trừ điểm câu sai.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import json
from src.data_loader import load_all
from src.utils import load_config, set_seed

set_seed(42)
cfg = load_config('../config.yaml')
cfg['paths']['dataset'] = '../dataset'
tables = load_all(cfg)

orders      = tables['orders']
order_items = tables['order_items']
products    = tables['products']
customers   = tables['customers']
geography   = tables['geography']
sales       = tables['sales']
returns     = tables['returns']
reviews     = tables['reviews']
payments    = tables['payments']
web_traffic = tables['web_traffic']
promotions  = tables['promotions']

answers = {}

[10:05:55] INFO data_loader: Loading all datasets...
c:\Users\admin\OneDrive - National Economics University\Documents\NCKH\DATATHON\Neu_BRT_Datathon\src\data_loader.py:50: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(_path(cfg, "order_items.csv"))
[10:05:56] INFO data_loader:   sales: 3,833 rows x 3 cols
[10:05:56] INFO data_loader:   submission: 548 rows x 3 cols
[10:05:56] INFO data_loader:   products: 2,412 rows x 8 cols
[10:05:56] INFO data_loader:   customers: 121,930 rows x 7 cols
[10:05:56] INFO data_loader:   geography: 39,948 rows x 4 cols
[10:05:56] INFO data_loader:   promotions: 50 rows x 10 cols
[10:05:56] INFO data_loader:   orders: 646,945 rows x 8 cols
[10:05:56] INFO data_loader:   order_items: 714,669 rows x 7 cols
[10:05:56] INFO data_loader:   payments: 646,945 rows x 4 cols
[10:05:56] INFO data_loader:   shipments: 566,067 rows x 4 cols
[10:05:56] INFO data_loader:   returns: 39,939 rows x

## Q1. Trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap)
Chỉ xét khách hàng có **nhiều hơn 1 đơn hàng**.

In [2]:
# Sort theo customer + date, tính diff
o = orders.sort_values(['customer_id', 'order_date'])
o['prev_date'] = o.groupby('customer_id')['order_date'].shift(1)
o['gap_days']  = (o['order_date'] - o['prev_date']).dt.days

# Chỉ lấy các khách có >1 đơn (gap_days không null = đơn thứ 2 trở đi)
gaps = o.dropna(subset=['gap_days'])

# Chỉ giữ những customer_id có ít nhất 1 gap (nghĩa là >1 đơn)
median_gap = gaps['gap_days'].median()
print(f'Trung vị inter-order gap: {median_gap:.1f} ngày')
print(f'Mean: {gaps["gap_days"].mean():.1f} | Std: {gaps["gap_days"].std():.1f}')
print(f'Distribution:\n{gaps["gap_days"].describe()}')

# Chọn đáp án
if median_gap < 60:     ans_q1 = 'A'
elif median_gap < 120:  ans_q1 = 'B'
elif median_gap < 250:  ans_q1 = 'C'
else:                   ans_q1 = 'D'
print(f'\n=> Đáp án Q1: {ans_q1}')
answers['Q1'] = ans_q1

Trung vị inter-order gap: 144.0 ngày
Mean: 285.6 | Std: 389.7
Distribution:
count    556699.000000
mean        285.592509
std         389.691558
min           0.000000
25%          46.000000
50%         144.000000
75%         357.000000
max        3785.000000
Name: gap_days, dtype: float64

=> Đáp án Q1: C


## Q2. Phân khúc (segment) có tỷ suất lợi nhuận gộp trung bình cao nhất
Công thức: **(price − cogs) / price**

In [3]:
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']
segment_margin = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print(segment_margin)

best_seg = segment_margin.idxmax()
print(f'\nPhân khúc có GM cao nhất: {best_seg} ({segment_margin.max():.4f})')

seg_map = {'Premium': 'A', 'Performance': 'B', 'Activewear': 'C', 'Standard': 'D'}
ans_q2 = seg_map.get(best_seg, 'A')
print(f'=> Đáp án Q2: {ans_q2}')
answers['Q2'] = ans_q2

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64

Phân khúc có GM cao nhất: Standard (0.3134)
=> Đáp án Q2: D


## Q3. Lý do trả hàng nhiều nhất trong danh mục **Streetwear**

In [4]:
# Join returns với products
ret_prod = returns.merge(products[['product_id', 'category']], on='product_id', how='left')
streetwear_returns = ret_prod[ret_prod['category'] == 'Streetwear']

reason_counts = streetwear_returns['return_reason'].value_counts()
print(f'Return reasons (Streetwear):\n{reason_counts}')

top_reason = reason_counts.idxmax()
print(f'\nLý do nhiều nhất: {top_reason}')

reason_map = {'defective': 'A', 'wrong_size': 'B', 'changed_mind': 'C', 'not_as_described': 'D'}
ans_q3 = reason_map.get(top_reason, 'A')
print(f'=> Đáp án Q3: {ans_q3}')
answers['Q3'] = ans_q3

Return reasons (Streetwear):
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

Lý do nhiều nhất: wrong_size
=> Đáp án Q3: B


## Q4. Nguồn traffic có tỷ lệ thoát (bounce_rate) trung bình **thấp nhất**

In [5]:
bounce_by_src = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print(f'Bounce rate trung bình theo nguồn:\n{bounce_by_src}')

lowest_src = bounce_by_src.idxmin()
print(f'\nNguồn bounce thấp nhất: {lowest_src} ({bounce_by_src.min():.4f})')

src_map = {'organic_search': 'A', 'paid_search': 'B', 'email_campaign': 'C', 'social_media': 'D'}
ans_q4 = src_map.get(lowest_src, 'A')
print(f'=> Đáp án Q4: {ans_q4}')
answers['Q4'] = ans_q4

Bounce rate trung bình theo nguồn:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

Nguồn bounce thấp nhất: email_campaign (0.0045)
=> Đáp án Q4: C


## Q5. % dòng order_items có áp dụng khuyến mãi (promo_id không null)

In [6]:
pct_promo = order_items['promo_id'].notna().mean() * 100
print(f'% dòng order_items có promo: {pct_promo:.2f}%')

if pct_promo < 18:    ans_q5 = 'A'
elif pct_promo < 32:  ans_q5 = 'B'
elif pct_promo < 46:  ans_q5 = 'C'
else:                 ans_q5 = 'D'
print(f'=> Đáp án Q5: {ans_q5}')
answers['Q5'] = ans_q5

% dòng order_items có promo: 38.66%
=> Đáp án Q5: C


## Q6. Nhóm tuổi có số đơn hàng trung bình **cao nhất** (tổng đơn / số khách)

In [7]:
# Join orders với customers
o_cust = orders.merge(customers[['customer_id', 'age_group']], on='customer_id', how='left')
o_cust = o_cust.dropna(subset=['age_group'])

# Tổng đơn mỗi nhóm
total_orders_group = o_cust.groupby('age_group')['order_id'].count()
# Số khách trong mỗi nhóm
n_customers_group = customers.dropna(subset=['age_group']).groupby('age_group')['customer_id'].count()

avg_orders = (total_orders_group / n_customers_group).sort_values(ascending=False)
print(f'Số đơn TB per khách theo nhóm tuổi:\n{avg_orders}')

best_group = avg_orders.idxmax()
print(f'\nNhóm tuổi nhiều đơn nhất: {best_group}')

group_map = {'55+': 'A', '25-34': 'B', '35-44': 'C', '45-54': 'D'}
ans_q6 = group_map.get(best_group, 'A')
print(f'=> Đáp án Q6: {ans_q6}')
answers['Q6'] = ans_q6

Số đơn TB per khách theo nhóm tuổi:
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
dtype: float64

Nhóm tuổi nhiều đơn nhất: 55+
=> Đáp án Q6: A


## Q7. Vùng (region) tạo tổng doanh thu cao nhất trong sales_train.csv
Lưu ý: sales.csv chỉ có Date/Revenue/COGS. Cần đi qua orders → geography → tính revenue theo region từ order_items.

In [8]:
# Join order_items -> orders -> geography để lấy region
oi_region = order_items.merge(
    orders[['order_id', 'order_date', 'zip']], on='order_id', how='left'
).merge(
    geography[['zip', 'region']], on='zip', how='left'
)

# Tính revenue: quantity * unit_price - discount_amount
oi_region['line_revenue'] = oi_region['quantity'] * oi_region['unit_price'] - oi_region['discount_amount'].fillna(0)

# Lọc chỉ train (<=2022-12-31)
train_end = pd.Timestamp('2022-12-31')
oi_train  = oi_region[oi_region['order_date'] <= train_end]

revenue_by_region = oi_train.groupby('region')['line_revenue'].sum().sort_values(ascending=False)
print(f'Doanh thu theo vùng:\n{revenue_by_region}')

best_region = revenue_by_region.idxmax()
print(f'\nVùng doanh thu cao nhất: {best_region}')

region_map = {'West': 'A', 'Central': 'B', 'East': 'C'}
# Kiểm tra nếu 3 vùng xấp xỉ bằng nhau (D)
if revenue_by_region.std() / revenue_by_region.mean() < 0.05:
    ans_q7 = 'D'
else:
    ans_q7 = region_map.get(best_region, 'D')
print(f'=> Đáp án Q7: {ans_q7}')
answers['Q7'] = ans_q7

Doanh thu theo vùng:
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: line_revenue, dtype: float64

Vùng doanh thu cao nhất: East
=> Đáp án Q7: C


## Q8. Phương thức thanh toán phổ biến nhất trong đơn hàng bị **cancelled**

In [9]:
cancelled = orders[orders['order_status'] == 'cancelled']
pm_cancelled = cancelled['payment_method'].value_counts()
print(f'Payment methods trong cancelled orders:\n{pm_cancelled}')

top_pm = pm_cancelled.idxmax()
print(f'\nPhổ biến nhất: {top_pm}')

pm_map = {'credit_card': 'A', 'cod': 'B', 'paypal': 'C', 'bank_transfer': 'D'}
ans_q8 = pm_map.get(top_pm, 'A')
print(f'=> Đáp án Q8: {ans_q8}')
answers['Q8'] = ans_q8

Payment methods trong cancelled orders:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

Phổ biến nhất: credit_card
=> Đáp án Q8: A


## Q9. Kích thước sản phẩm (S/M/L/XL) có **tỷ lệ trả hàng cao nhất**
Tỷ lệ = số bản ghi returns / số dòng order_items (join với products theo product_id)

In [10]:
# Số dòng order_items theo size
oi_size = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')
oi_size = oi_size[oi_size['size'].isin(['S', 'M', 'L', 'XL'])]
oi_count = oi_size.groupby('size')['order_id'].count()

# Số bản ghi returns theo size
ret_size = returns.merge(products[['product_id', 'size']], on='product_id', how='left')
ret_size = ret_size[ret_size['size'].isin(['S', 'M', 'L', 'XL'])]
ret_count = ret_size.groupby('size')['return_id'].count()

return_rate = (ret_count / oi_count).sort_values(ascending=False)
print(f'Tỷ lệ trả hàng theo size:\n{return_rate}')

top_size = return_rate.idxmax()
print(f'\nSize tỷ lệ trả cao nhất: {top_size}')

size_map = {'S': 'A', 'M': 'B', 'L': 'C', 'XL': 'D'}
ans_q9 = size_map.get(top_size, 'A')
print(f'=> Đáp án Q9: {ans_q9}')
answers['Q9'] = ans_q9

Tỷ lệ trả hàng theo size:
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64

Size tỷ lệ trả cao nhất: S
=> Đáp án Q9: A


## Q10. Kế hoạch trả góp có giá trị thanh toán trung bình **cao nhất**

In [11]:
avg_by_installment = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print(f'Giá trị thanh toán TB theo số kỳ:\n{avg_by_installment}')

top_inst = avg_by_installment.idxmax()
print(f'\nSố kỳ có giá trị TB cao nhất: {top_inst}')

inst_map = {1: 'A', 3: 'B', 6: 'C', 12: 'D'}
ans_q10 = inst_map.get(top_inst, 'D')
print(f'=> Đáp án Q10: {ans_q10}')
answers['Q10'] = ans_q10

Giá trị thanh toán TB theo số kỳ:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64

Số kỳ có giá trị TB cao nhất: 6
=> Đáp án Q10: C


## Tổng hợp đáp án

In [12]:
print('='*40)
print('TỔNG HỢP ĐÁP ÁN MCQ')
print('='*40)
for q, a in sorted(answers.items()):
    print(f'{q}: {a}')

# Lưu ra file
import os
os.makedirs('../outputs', exist_ok=True)
with open('../outputs/mcq_answers.json', 'w', encoding='utf-8') as f:
    json.dump(answers, f, ensure_ascii=False, indent=2)
print('\nSaved to ../outputs/mcq_answers.json')

TỔNG HỢP ĐÁP ÁN MCQ
Q1: C
Q10: C
Q2: D
Q3: B
Q4: C
Q5: C
Q6: A
Q7: C
Q8: A
Q9: A

Saved to ../outputs/mcq_answers.json
